In [ ]:
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import os
import random

In [ ]:
projects_list = pd.read_csv("energyconsents_df_with_links.csv")
pd.set_option("display.max_colwidth", None)
projects_list.head()

,ECU Reference,Project Name,Case Type,Project Type,Case Status Type,Url
0,ECU00003354,Tealing Battery Energy Storage farm,Development,Battery Energy Storage System,Consented,https://www.energyconsents.scot/ApplicationDetails.aspx?cr=ECU00003354&T=4
1,ECU00003435,Kilmarnock South Battery Electricity Storage System (BESS),Development,Battery Energy Storage System,Consented,https://www.energyconsents.scot/ApplicationDetails.aspx?cr=ECU00003435&T=4
2,ECU00003458,Coalburn Energy Storage Project(ECU36COALBURN),Development,Battery Energy Storage System,Consented,https://www.energyconsents.scot/ApplicationDetails.aspx?cr=ECU00003458&T=4
3,ECU00003469,Devilla Energy Storage Project,Development,Battery Energy Storage System,Consented,https://www.energyconsents.scot/ApplicationDetails.aspx?cr=ECU00003469&T=4
4,ECU00004491,"Keithick, Stirling 50.1MW BESS",Development,Battery Energy Storage System,Withdrawn,https://www.energyconsents.scot/ApplicationDetails.aspx?cr=ECU00004491&T=4


In [ ]:
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (thesis-scraper/1.0)"})

In [ ]:
def extract_hidden(soup):
    form = soup.find("form")
    payload = {}   #extracting the hidden fields - needed to access the main project site
    for inp in form.find_all("input", {"type":"hidden"}):
        name = inp.get("name")
        if name:
            payload[name] = inp.get("value", "")
    return payload

In [ ]:
#finding the next page button
def next_button(soup):
    a_next = soup.find("a", title = lambda t:t and t.startswith("Next Page"))
    if not a_next:
        return None
    if "aspNetDisabled" in (a_next.get("class") or []):
        return None

#extracting postback

    href = a_next.get("href", "")
    m = re.search(r"__doPostBack\('(.+?)','(.*?)'\)", href)
    if not m:
        return None

    return m.group(1), m.group(2)

In [ ]:

def scrape_one_project(session: requests.Session, url, project_ref, base_folder= "projects_downloads_all"):
    #downloads all public representations for one project, across all pages
    results = []

    project_folder = os.path.join(base_folder, str(project_ref))
    os.makedirs(project_folder, exist_ok = True)

    r_main = session.get(url,headers={"Referer": url},timeout=20)

    if r_main.status_code != 200:
        print("No  public representations page (http error) for:", project_ref, r_main.status_code)
        results.append({
            "project_ref": project_ref,
            "url": url,
            "status": "http_error_representations",
            "http": r_main.status_code
        })
        return results
    print("Public representations page loaded for:", project_ref)

#for dealing with reaching documents page first (main page as a starting point), and from there going to representations
    soup1 = BeautifulSoup(r_main.text, "html.parser")
    payload1 = extract_hidden(soup1)

    EVENT_DOCUMENTS = "ctl00$ContentPlaceHolder1$htpDocuments"
    payload1["__EVENTTARGET"] = EVENT_DOCUMENTS
    payload1["__EVENTARGUMENT"] = ""

    r_documents = session.post(
        url,
        data=payload1,
        headers={"Referer": url},
        timeout=30
    )


    r_representations = session.get(url,headers={"Referer": url},timeout=20)

    if r_representations.status_code != 200:
        print("No  public representations page (http error) for:", project_ref, r_representations.status_code)
        results.append({
            "project_ref": project_ref,
            "url": url,
            "status": "http_error_representations",
            "http": r_representations.status_code
        })
        return results



    page_html = r_representations.text
    downloaded_any = False


    while True:
        soup_representations = BeautifulSoup(page_html, "html.parser")
        payload2 = extract_hidden(soup_representations)

        pdf_links = soup_representations.find_all("a", class_="ns") #every comments' link has class ns
        if not pdf_links:
            print("No pdf postback links found on this page:", project_ref)
            if not downloaded_any:
                 results.append({
                    "project_ref": project_ref,
                    "url": url,
                    "status": "no_comments"
                })
            break


        for i, a in enumerate(pdf_links,start=1):
            href = a.get("href", "")
            m = re.search(r"__doPostBack\('(.+?)','(.*?)'\)", href)
            if not m:
                continue

            event_target = m.group(1)
            event_argument = m.group(2)

            filename_txt = a.get_text(strip=True)
            filename = (f"{project_ref} {filename_txt}.pdf")
            filepath = os.path.join(project_folder, filename)
            if os.path.exists(filepath):
                results.append({
                    "project_ref": project_ref,
                    "url": url,
                    "status": "skipped_exists",
                    "file": filepath
                })
                print("  -> Already exists, skipping:", filename)
                continue


            payload2_copy = dict(payload2) #for starting each post
            payload2_copy["__EVENTTARGET"] = event_target
            payload2_copy["__EVENTARGUMENT"] = event_argument



              #post back to the same representations url
            pdf_resp = session.post(
                url,
                data=payload2_copy,
                headers={"Referer": url},
                timeout=60)
            time.sleep(random.uniform(0.8, 1.5))


            if pdf_resp.content[:4] != b"%PDF":
                print("Not a PDF:", project_ref, filename_txt, pdf_resp.status_code, pdf_resp.headers.get("Content-Type"))
                results.append({
                    "project_ref": project_ref,
                    "url": url,
                    "status": "not_pdf",
                    "http": pdf_resp.status_code,
                    "content_type": pdf_resp.headers.get("Content-Type"),
                    "filename": filename
                })
                continue



            with open(filepath, "wb") as f:
              f.write(pdf_resp.content)

            downloaded_any = True
            results.append({
                "project_ref": project_ref,
                "url": url,
                "status": "downloaded",
                "file": filepath,
                "http": pdf_resp.status_code
            })

            #for tracking the download
            print(i, pdf_resp.status_code, pdf_resp.headers.get("Content-Type"))
            print(pdf_resp.content[:4])

        next_page = next_button(soup_representations)
        if not next_page:
            print("no next page, done")
            break

        next_target, next_argument =next_page
        payload_next = extract_hidden(soup_representations)
        payload_next["__EVENTTARGET"] = next_target
        payload_next["__EVENTARGUMENT"] = next_argument
        r_next = session.post(
            url,
            data = payload_next,
            headers={"Referer":url},
            timeout=60)
        time.sleep(random.uniform(0.8, 1.5))
        page_html = r_next.text

    return results

In [ ]:
#interating over every url
all_results = []

for idx, row in projects_list.iterrows():
    url = row["Url"]
    project_ref = row["ECU Reference"]

    print(f"[{idx+1}/{len(projects_list)}] {project_ref}")

    try:
        all_results.extend(scrape_one_project(session, url, project_ref))
    except Exception as e:
        all_results.append({
            "project_ref": project_ref,
            "url": url,
            "status": "exception",
            "error": repr(e)
        })

    if (idx + 1) % 25 == 0:  #checkpoint
        pd.DataFrame(all_results).to_csv("download_log.csv", index=False)

log_df = pd.DataFrame(all_results)
log_df.to_csv("download_log.csv", index=False)

[1/94] ECU00003354
Public representations page loaded for: ECU00003354
  -> Already exists, skipping: ECU00003354 Representation 001 020 Objection - Redacted.pdf
  -> Already exists, skipping: ECU00003354 Representation 021 043 Objection - Redacted.pdf
  -> Already exists, skipping: ECU00003354 Representation 044 050 Objection - Redacted.pdf
  -> Already exists, skipping: ECU00003354 Representation Redacted 051 054 Objection.pdf
  -> Already exists, skipping: ECU00003354 Representation Redacted 055 Objection.pdf
  -> Already exists, skipping: ECU00003354 Representation Redacted 056 Objection.pdf
  -> Already exists, skipping: ECU00003354 Representation Redacted 057 Objection.pdf
  -> Already exists, skipping: ECU00003354 Representation Redacted 058 059 Objection.pdf
  -> Already exists, skipping: ECU00003354 Representation Redacted 058 - SSEN Transmission - No Objection.pdf
  -> Already exists, skipping: ECU00003354 Further Comments to Representation 0051 Objection - Tealing Battery En